# TIDES × Original EDS — GPU run & results dashboard

**Sequence:** `00_peanuts_dark` · **Original EDS** (not the EvTween preprocessed copy). Run cells **in order**. Each expensive operation has its own explicit switch; no shell commands are pasted into Python cells. Training and ablations are disabled by default.

**Scientific status:** independent TIDES implementation; a 1.5-second single-sequence test is **not** the paper's dataset-level reproduction. The pose-reference frame and exact paper protocol still require verification.

In [ ]:
from pathlib import Path
import os, sys, json, shlex, subprocess, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

here = Path.cwd().resolve()
candidates = [here, *here.parents, Path('/u/marmary/airlab_v2e/TIDES_2026')]
ROOT = next((p for p in candidates if (p/'scripts'/'prepare_scene.py').exists()), None)
assert ROOT is not None, f'Cannot locate TIDES_2026 repository from {here}'

def run(*args):
    cmd = [str(x) for x in args]
    print('$', shlex.join(cmd), flush=True)
    subprocess.run(cmd, cwd=ROOT, env=os.environ.copy(), check=True)

print('Repository:', ROOT)
print('Python:', sys.executable)

## 1. Paths, calibration, and run controls
Run this cell after changing any setting. Keep **only the next stage** enabled.

In [ ]:
EDS_ROOT = Path('/u/marmary/evtween/EDS/EDS')
SEQUENCE = '00_peanuts_dark'
CALIB_FILE = Path('/u/marmary/evtween/EDS/calib/00_calib/camchain-mediajaviJAVISdatasetshwdscalibratione2kalibr.yaml')
WORK = ROOT/'work'/'eds'/SEQUENCE
FOURDGS = ROOT/'third_party'/'4DGaussians'
T_START, T_END = 0.0, 1.5
SEED = 0
PORT = 6017

# Enable one stage at a time. Existing prepared scene is reused by default.
DO_INSPECT = False
DO_PREPARE = False
DO_TRAIN = False      # Set True ONLY when ready for expensive GPU training
DO_TIDES = False
DO_BASELINES = False
DO_EVALUATE = False
DO_ABLATIONS = False

TIDES_METHODS = ['tides_10x', 'tides_adaptive']
BASELINE_METHODS = ['esim','v2e','iebcs','dvs_voltmeter']

assert EDS_ROOT.is_dir(), f'EDS root missing: {EDS_ROOT}'
assert CALIB_FILE.is_file(), f'Calibration missing: {CALIB_FILE}'
os.environ['TIDES_EDS_CALIB'] = str(CALIB_FILE)
print('Calibration:', CALIB_FILE)
print('Work:', WORK)
print('Prepared:', (WORK/'tides_meta.json').exists())
print('4DGS train script:', (FOURDGS/'train.py').exists())
print('4DGS checkpoint:', (WORK/'4dgs'/'point_cloud').exists())
print('Enabled:', [k for k in ['DO_INSPECT','DO_PREPARE','DO_TRAIN','DO_TIDES','DO_BASELINES','DO_EVALUATE','DO_ABLATIONS'] if globals()[k]])

## 2. Validate original EDS / prepare scene
The original EDS contains RGB images, real events, ground-truth trajectories, and a separate Kalibr calibration. `DO_PREPARE=True` regenerates the scene; keep it False to reuse the 113-frame scene already prepared.

In [ ]:
if DO_INSPECT:
    run(sys.executable, ROOT/'scripts/inspect_dataset.py', '--dataset','eds','--root',EDS_ROOT,'--seq',SEQUENCE)
if DO_PREPARE:
    run(sys.executable, ROOT/'scripts/prepare_scene.py', '--dataset','eds','--root',EDS_ROOT,'--seq',SEQUENCE,'--out',WORK,'--t-start',T_START,'--t-end',T_END)
if not DO_INSPECT and not DO_PREPARE:
    print('Inspection/preparation skipped; existing files are unchanged.')
meta_file=WORK/'tides_meta.json'
if meta_file.exists():
    meta=json.loads(meta_file.read_text())
    for key in ['dataset','sequence','t_range','frame_interval','K_train','K_render','scale']:
        print(key,':',meta.get(key))
    print('Frames:',len(meta.get('frame_times_s',[])))
else:
    print('No prepared metadata yet.')

## 3. Train 4D Gaussian Splatting — GPU
**Only run after explicitly setting `DO_TRAIN=True` in Settings and rerunning that cell.** This may take substantial time. Existing checkpoint is never silently overwritten by this notebook.

In [ ]:
checkpoint_dir = WORK/'4dgs'/'point_cloud'
if DO_TRAIN:
    assert (WORK/'scene'/'images').exists(), 'Prepared scene missing'
    assert (FOURDGS/'train.py').is_file(), f'4DGaussians checkout missing: {FOURDGS}'
    assert not checkpoint_dir.exists(), 'Checkpoint already exists; training stopped to avoid overwriting it.'
    import torch
    assert torch.cuda.is_available(), 'GPU not available in current notebook kernel'
    print('GPU:',torch.cuda.get_device_name(0))
    run(sys.executable, FOURDGS/'train.py','-s',WORK/'scene','-m',WORK/'4dgs',
        '--configs',ROOT/'configs'/'4dgs'/'tides_default.py','--port',PORT)
else:
    print('Training disabled. To train: DO_TRAIN=True, rerun Settings, then rerun this cell.')
print('Checkpoint directory:',checkpoint_dir,'exists:',checkpoint_dir.exists())

## 4. Simulate TIDES events
Requires a completed 4DGS checkpoint. The two variants write separate HDF5 event files.

In [ ]:
if DO_TIDES:
    assert checkpoint_dir.exists(), 'No 4DGS checkpoint; complete training first'
    for name in TIDES_METHODS:
        output=WORK/'events'/f'{name}.h5'
        if output.exists():
            print('Reusing:',output)
            continue
        run(sys.executable,ROOT/'scripts'/'simulate.py','--work',WORK,'--fourdgs',FOURDGS,
            '--config',ROOT/'configs'/f'{name}.yaml','--name',name,'--seed',SEED)
else:
    print('TIDES simulation disabled.')
for name in TIDES_METHODS:
    f=WORK/'events'/f'{name}.h5'
    print(name, 'READY' if f.exists() else 'not generated')

## 5. Optional baselines
ESIM, V2E, IEBCS, and DVS-Voltmeter may require additional dependencies. This stage can be expensive.

In [ ]:
if DO_BASELINES:
    assert checkpoint_dir.exists(), 'No trained checkpoint'
    run(sys.executable,ROOT/'scripts'/'run_baselines.py','--work',WORK,'--fourdgs',FOURDGS,
        '--third-party',ROOT/'third_party','--methods',*BASELINE_METHODS)
else:
    print('Baselines disabled.')

## 6. Evaluate IG-NLL and Chamfer
Scores only existing generated event files against the original EDS real events. The repository metric settings are assumptions and may differ from the paper.

In [ ]:
all_methods=TIDES_METHODS+BASELINE_METHODS
available=[m for m in all_methods if (WORK/'events'/f'{m}.h5').exists()]
print('Available event files:',available)
if DO_EVALUATE:
    assert available, 'Generate events first'
    run(sys.executable,ROOT/'scripts'/'evaluate.py','--work',WORK,'--fourdgs',FOURDGS,'--methods',*available)
else:
    print('Evaluation disabled.')
metrics_file=WORK/'metrics.json'
results=json.loads(metrics_file.read_text()) if metrics_file.exists() else {}
print('Metrics file:',metrics_file,'exists:',metrics_file.exists())

## 7. Results dashboard — safe to rerun anytime
Displays current artifacts, metric table, charts, and run metadata. Missing results are clearly labeled rather than fabricated.

In [ ]:
rows=[]
for m in all_methods:
    f=WORK/'events'/f'{m}.h5'
    values=results.get('methods',{}).get(m,{})
    rows.append({'method':m,'events_file':f.exists(),'IG-NLL':values.get('ig_nll',np.nan),
                 'Chamfer':values.get('chamfer',np.nan), 'file_MB':round(f.stat().st_size/1e6,1) if f.exists() else np.nan})
df=pd.DataFrame(rows)
display(df)
fig, axes=plt.subplots(1,2,figsize=(12,4))
for ax,col in zip(axes,['IG-NLL','Chamfer']):
    good=df.dropna(subset=[col])
    if len(good):
        ax.bar(good['method'],good[col]);ax.tick_params(axis='x',rotation=35)
    else: ax.text(.5,.5,'No evaluated results yet',ha='center',va='center',transform=ax.transAxes)
    ax.set_title(col);ax.set_ylabel(col)
plt.tight_layout();plt.show()

## 8. Inspect event-file structure and run statistics
The HDF5 structure is inspected without loading millions of events into memory.

In [ ]:
import h5py
for m in TIDES_METHODS:
    f=WORK/'events'/f'{m}.h5'
    if not f.exists():continue
    print('\n',m, f'({f.stat().st_size/1e6:.1f} MB)')
    with h5py.File(f,'r') as h:
        def show(name,obj):
            if isinstance(obj,h5py.Dataset): print(' ',name,'shape=',obj.shape,'dtype=',obj.dtype)
        h.visititems(show)
    stats=WORK/'events'/f'{m}.json'
    if stats.exists():
        info=json.loads(stats.read_text())
        print('Stats:',json.dumps(info.get('stats',{}),indent=2,default=str)[:2500])

## 9. Paper reference — contextual only
Paper values are EDS **dataset averages**, whereas this notebook evaluates one short sequence. Differences cannot be interpreted as an exact reproduction gap.

In [ ]:
try:
    from tides.paper_numbers import TABLE1, TABLE4
    compare=[]
    for m in TIDES_METHODS+BASELINE_METHODS:
        vals=results.get('methods',{}).get(m,{})
        if m in TABLE1 and 'eds' in TABLE1[m]:
            paper_ig,paper_ch=TABLE1[m]['eds']
            compare.append({'method':m,'our IG-NLL':vals.get('ig_nll'), 'paper EDS mean IG-NLL':paper_ig,
                            'our Chamfer':vals.get('chamfer'),'paper EDS mean Chamfer':paper_ch})
    display(pd.DataFrame(compare))
    print('Paper Table 4 reference (repository transcription):')
    display(pd.DataFrame([{'variant':k,'label':v[0],'IG-NLL':v[1],'Chamfer':v[2]} for k,v in TABLE4.items()]))
except Exception as exc:
    print('Paper comparison unavailable:',type(exc).__name__,str(exc))

## 10. Optional ablations and run manifest
Do not run ablations until TIDES simulation and evaluation work reliably.

In [ ]:
if DO_ABLATIONS:
    assert checkpoint_dir.exists(), 'No 4DGS checkpoint'
    run(sys.executable,ROOT/'scripts'/'run_ablations.py','--work',WORK,'--fourdgs',FOURDGS)
else:
    print('Ablations disabled.')
manifest={'dataset_root':str(EDS_ROOT),'sequence':SEQUENCE,'calibration':str(CALIB_FILE),
          'work':str(WORK),'time_window_requested':[T_START,T_END],'seed':SEED,
          'checkpoint_exists':checkpoint_dir.exists(),'event_methods_available':available,
          'metrics_file_exists':metrics_file.exists(),
          'limitations':['Pose reference frame not independently validated',
                         'Independent implementation and metric settings may differ from paper',
                         'One short sequence cannot reproduce EDS dataset mean']}
if WORK.exists():
    out=WORK/'notebook_run_manifest.json';out.write_text(json.dumps(manifest,indent=2));print('Manifest:',out)
display(pd.DataFrame([manifest]).T.rename(columns={0:'value'}))